In [ ]:
#@title 🎙️ **an8nymous TTS** - Clique em ▶️ para iniciar
#@markdown ---
#@markdown ### Clonagem de Voz com Inteligência Artificial
#@markdown
#@markdown **Instruções:**
#@markdown 1. Clique no botão ▶️ (play) à esquerda
#@markdown 2. Autorize o acesso ao Google Drive (para cache)
#@markdown 3. Aguarde (1ª vez: ~3 min | próximas: ~30 seg)
#@markdown 4. Acesse a URL pública gerada
#@markdown
#@markdown ---

import time
START_TIME = time.time()

print("="*60)
print("🎙️ an8nymous TTS - Clonagem de Voz com IA")
print("="*60)
print()

# ============================================================
# VERIFICAR GPU
# ============================================================

import torch
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    print(f"✅ GPU detectada: {gpu}")
else:
    print("⚠️ GPU não disponível! Vá em Runtime > Change runtime type > T4 GPU")
    print("   Sem GPU, a geração será MUITO lenta.")

# ============================================================
# MONTAR GOOGLE DRIVE (para cache do modelo)
# ============================================================

print()
print("📁 Montando Google Drive para cache...")

from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import os
import sys

# Diretório de cache no Drive (APENAS para modelo HuggingFace)
CACHE_BASE = '/content/drive/MyDrive/an8nymous_tts_cache'
CACHE_MODELS = f'{CACHE_BASE}/models'

os.makedirs(CACHE_MODELS, exist_ok=True)

# Configurar HuggingFace para usar cache no Drive
os.environ['HF_HOME'] = CACHE_MODELS
os.environ['TRANSFORMERS_CACHE'] = CACHE_MODELS
os.environ['HF_DATASETS_CACHE'] = CACHE_MODELS

print(f"✅ Cache do modelo: {CACHE_MODELS}")

# ============================================================
# FIX: Forçar numpy 1.26.x (Colab vem com numpy 2.x incompatível)
# ============================================================

print()
print("🔧 Verificando numpy...")
import subprocess
result = subprocess.run(['pip', 'show', 'numpy'], capture_output=True, text=True)
current_numpy = [l for l in result.stdout.splitlines() if l.startswith('Version:')]
if current_numpy:
    version = current_numpy[0].split(':')[1].strip()
    if version.startswith('2.'):
        print(f"   ⚠️ numpy {version} detectado (incompatível)")
        print("   📥 Instalando numpy 1.26.x...")
        !pip uninstall -y numpy -q 2>/dev/null
        !pip install -q numpy==1.26.4 2>/dev/null
        print("   ✅ numpy 1.26.4 instalado!")
        print("   🔄 Reiniciando runtime...")
        os.kill(os.getpid(), 9)
    else:
        print(f"   ✅ numpy {version} OK")

# ============================================================
# INSTALAÇÃO DE DEPENDÊNCIAS (global - mais rápido e estável)
# ============================================================

DEPS_MARKER = '/content/.deps_installed_v4'

if os.path.exists(DEPS_MARKER):
    print("⚡ Dependências já instaladas nesta sessão!")
else:
    print("🔧 Instalando dependências...")
    
    # Pacotes necessários (instalação global - sem conflitos)
    !pip install -q resampy==0.4.3 librosa==0.11.0 2>/dev/null
    !pip install -q s3tokenizer==0.2.0 2>/dev/null
    !pip install -q transformers==4.46.3 diffusers==0.29.0 2>/dev/null
    !pip install -q resemble-perth==1.0.1 omegaconf==2.3.0 2>/dev/null
    !pip install -q conformer==0.3.2 safetensors==0.5.3 2>/dev/null
    !pip install -q sentencex==0.6.1 pydub==0.25.1 soundfile==0.13.1 2>/dev/null
    !pip install -q gradio>=4.0.0 2>/dev/null
    
    # Marcar como instalado
    with open(DEPS_MARKER, 'w') as f:
        f.write('installed')
    
    print("✅ Dependências instaladas!")

# ============================================================
# CLONAR REPOSITÓRIO
# ============================================================

REPO_PATH = '/content/chatterbox'

if os.path.exists(f'{REPO_PATH}/src/chatterbox'):
    print("⚡ Repositório já clonado!")
else:
    print("📥 Clonando repositório...")
    !rm -rf {REPO_PATH} 2>/dev/null
    !git clone -q https://github.com/resemble-ai/chatterbox.git {REPO_PATH} 2>/dev/null
    
    # Fix para multilingual
    !wget -q -O {REPO_PATH}/src/chatterbox/__init__.py \
      https://raw.githubusercontent.com/NeuralFalconYT/Chatterbox-Multilingual/main/__init__.py 2>/dev/null
    
    print("✅ Repositório clonado!")

sys.path.insert(0, f'{REPO_PATH}/src')

# ============================================================
# CONFIGURAÇÃO
# ============================================================

import warnings
warnings.filterwarnings("ignore")

os.makedirs("./audios_gerados", exist_ok=True)

# ============================================================
# CARREGAR MODELO (cacheado no Drive)
# ============================================================

print()
print("🔄 Carregando modelo de IA...")

MODEL_CACHE_CHECK = os.path.join(CACHE_MODELS, 'hub')
if os.path.exists(MODEL_CACHE_CHECK) and os.listdir(MODEL_CACHE_CHECK):
    print("   ⚡ Modelo encontrado no cache!")
else:
    print("   📥 Baixando modelo (~2GB, 1ª vez apenas)...")

from chatterbox.mtl_tts import ChatterboxMultilingualTTS
import numpy as np
import soundfile as sf
import uuid
import re
import random
from tqdm.auto import tqdm

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL = ChatterboxMultilingualTTS.from_pretrained(DEVICE)

LOAD_TIME = time.time() - START_TIME
print(f"✅ Modelo carregado no {DEVICE.upper()}! ({LOAD_TIME:.1f}s)")
print()

# ============================================================
# CONFIGURAÇÕES
# ============================================================

IDIOMAS = {
    "Português": "pt",
    "Inglês": "en",
    "Espanhol": "es",
    "Francês": "fr",
    "Alemão": "de",
    "Italiano": "it",
    "Japonês": "ja",
    "Chinês": "zh",
    "Coreano": "ko",
    "Russo": "ru",
    "Árabe": "ar",
    "Hindi": "hi",
    "Holandês": "nl",
    "Polonês": "pl",
    "Turco": "tr",
    "Sueco": "sv",
    "Dinamarquês": "da",
    "Norueguês": "no",
    "Finlandês": "fi",
    "Grego": "el",
    "Hebraico": "he",
    "Malaio": "ms",
    "Suaíli": "sw",
}

VOZES_PADRAO = {
    "pt": "https://storage.googleapis.com/chatterbox-demo-samples/mtl_prompts/pt_m1.flac",
    "en": "https://storage.googleapis.com/chatterbox-demo-samples/mtl_prompts/en_f1.flac",
    "es": "https://storage.googleapis.com/chatterbox-demo-samples/mtl_prompts/es_f1.flac",
    "fr": "https://storage.googleapis.com/chatterbox-demo-samples/mtl_prompts/fr_f1.flac",
    "de": "https://storage.googleapis.com/chatterbox-demo-samples/mtl_prompts/de_f1.flac",
    "it": "https://storage.googleapis.com/chatterbox-demo-samples/mtl_prompts/it_m1.flac",
    "ja": "https://storage.googleapis.com/chatterbox-demo-samples/mtl_prompts/ja/ja_prompts1.flac",
    "zh": "https://storage.googleapis.com/chatterbox-demo-samples/mtl_prompts/zh_f2.flac",
    "ko": "https://storage.googleapis.com/chatterbox-demo-samples/mtl_prompts/ko_f.flac",
    "ru": "https://storage.googleapis.com/chatterbox-demo-samples/mtl_prompts/ru_m.flac",
    "ar": "https://storage.googleapis.com/chatterbox-demo-samples/mtl_prompts/ar_f/ar_prompts2.flac",
    "hi": "https://storage.googleapis.com/chatterbox-demo-samples/mtl_prompts/hi_f1.flac",
}

TEXTOS_EXEMPLO = {
    "pt": "Olá! Este é um teste de clonagem de voz em português brasileiro. A qualidade está incrível!",
    "en": "Hello! This is a voice cloning test in English. The quality is amazing!",
    "es": "¡Hola! Esta es una prueba de clonación de voz en español.",
}

# ============================================================
# FUNÇÃO DE GERAÇÃO
# ============================================================

def gerar_audio(texto, idioma, audio_ref, expressividade, temperatura, cfg_peso, seed, remover_silencio):
    """Gera áudio a partir de texto"""
    from sentencex import segment

    codigo = IDIOMAS.get(idioma, "pt")

    if not audio_ref or not os.path.exists(str(audio_ref)):
        audio_ref = VOZES_PADRAO.get(codigo, VOZES_PADRAO["pt"])

    if seed == 0:
        seed = random.randint(1, 999999)
    torch.manual_seed(seed)
    random.seed(seed)
    np.random.seed(seed)

    texto = re.sub(r'[\*\#\-\—\–]', ' ', texto)
    texto = re.sub(r'\s+', ' ', texto).strip()

    if len(texto) > 300:
        chunks = []
        for sent in segment(codigo, texto):
            if len(sent) <= 300:
                chunks.append(sent)
            else:
                words = sent.split()
                current = ""
                for word in words:
                    if len(current) + len(word) + 1 <= 300:
                        current += (" " if current else "") + word
                    else:
                        chunks.append(current)
                        current = word
                if current:
                    chunks.append(current)
    else:
        chunks = [texto]

    audios = []
    for chunk in tqdm(chunks, desc="Gerando"):
        try:
            wav = MODEL.generate(
                chunk,
                language_id=codigo,
                audio_prompt_path=audio_ref,
                exaggeration=expressividade,
                temperature=temperatura,
                cfg_weight=cfg_peso,
            )
            audios.append(wav.squeeze(0).numpy())
        except Exception as e:
            print(f"⚠️ Erro: {e}")

    if not audios:
        raise RuntimeError("Falha ao gerar áudio")

    audio_final = np.concatenate(audios)

    nome = re.sub(r'[^a-zA-Z\s]', '', texto[:20]).lower().replace(" ", "_")
    nome = f"./audios_gerados/{nome}_{codigo}_{uuid.uuid4().hex[:6]}.wav"
    sf.write(nome, audio_final, MODEL.sr, subtype='PCM_24')

    if remover_silencio:
        from pydub import AudioSegment
        from pydub.silence import split_on_silence
        sound = AudioSegment.from_file(nome)
        chunks = split_on_silence(sound, min_silence_len=100, silence_thresh=-45, keep_silence=50)
        combined = AudioSegment.empty()
        for c in chunks:
            combined += c
        nome_limpo = nome.replace(".wav", "_limpo.wav")
        combined.export(nome_limpo, format="wav")
        nome = nome_limpo

    return nome, nome

# ============================================================
# INTERFACE GRADIO
# ============================================================

import gradio as gr

css = """
.gradio-container { max-width: 900px !important; margin: auto !important; }
.header { text-align: center; padding: 20px; background: linear-gradient(135deg, #667eea, #764ba2); border-radius: 12px; margin-bottom: 20px; }
.header h1 { color: white; margin: 0; font-size: 2em; }
.header p { color: rgba(255,255,255,0.9); margin: 5px 0 0 0; }
.footer { text-align: center; padding: 15px; color: #666; font-size: 0.85em; border-top: 1px solid #eee; margin-top: 20px; }
.status-box { background: #1a1a2e; border: 1px solid #333; border-radius: 8px; padding: 10px; margin: 10px 0; }
.status-box p { margin: 2px 0; font-size: 0.85em; }
"""

with gr.Blocks(theme=gr.themes.Soft(), css=css, title="an8nymous TTS") as demo:

    gr.HTML('<div class="header"><h1>🎙️ an8nymous TTS</h1><p>Clonagem de Voz com Inteligência Artificial</p></div>')

    with gr.Row():
        with gr.Column():
            texto = gr.Textbox(value=TEXTOS_EXEMPLO["pt"], label="📝 Texto", lines=4)
            idioma = gr.Dropdown(choices=list(IDIOMAS.keys()), value="Português", label="🌍 Idioma (23 disponíveis)")
            audio_ref = gr.Audio(sources=["upload", "microphone"], type="filepath",
                                 label="🎤 Áudio de Referência (6-10s)", value=VOZES_PADRAO["pt"])
            btn = gr.Button("🚀 Gerar Áudio", variant="primary", size="lg")

            with gr.Accordion("⚙️ Opções Avançadas", open=False):
                expressividade = gr.Slider(0.25, 2.0, value=0.5, label="Expressividade (0.5=neutro)")
                temperatura = gr.Slider(0.05, 2.0, value=0.8, label="Temperatura")
                cfg_peso = gr.Slider(0.2, 1.0, value=0.5, label="CFG/Ritmo")
                seed = gr.Number(value=0, label="Seed (0=aleatório)")
                remover_silencio = gr.Checkbox(value=False, label="Remover silêncios")

        with gr.Column():
            audio_out = gr.Audio(label="🔊 Áudio Gerado")
            arquivo = gr.File(label="📥 Download")

    gr.HTML(f'''
    <div class="status-box">
        <p>⚡ <b>Modelo cacheado</b>: Google Drive/an8nymous_tts_cache/models</p>
        <p>⏱️ <b>Tempo de inicialização</b>: {LOAD_TIME:.1f} segundos</p>
    </div>
    ''')

    gr.HTML('<div class="footer"><p><b>an8nymous TTS</b> - Powered by <a href="https://github.com/resemble-ai/chatterbox">Chatterbox</a> (Resemble AI)<br>Adaptação: <a href="https://github.com/NeuralFalconYT/Chatterbox-Multilingual">NeuralFalconYT</a></p><p style="font-size:0.8em;color:#999">⚠️ Use com responsabilidade. Não clone vozes sem autorização.</p></div>')

    def ao_mudar_idioma(idioma):
        codigo = IDIOMAS.get(idioma, "pt")
        return TEXTOS_EXEMPLO.get(codigo, TEXTOS_EXEMPLO["pt"]), VOZES_PADRAO.get(codigo, VOZES_PADRAO["pt"])

    idioma.change(ao_mudar_idioma, [idioma], [texto, audio_ref])
    btn.click(gerar_audio, [texto, idioma, audio_ref, expressividade, temperatura, cfg_peso, seed, remover_silencio], [audio_out, arquivo])

# ============================================================
# KEEP-ALIVE
# ============================================================

from IPython.display import display, Javascript, Audio

keep_alive_js = Javascript('''
function KeepAlive() {
    console.log("[an8nymous] Keep-alive ping");
    var btn = document.querySelector("colab-connect-button");
    if (btn) btn.click();
}
setInterval(KeepAlive, 60000);
''')

SILENCE_AUDIO_URL = "https://raw.githubusercontent.com/KoboldAI/KoboldAI-Client/main/colab/silence.m4a"

print("🛡️ Keep-alive ativado")
display(keep_alive_js)
display(Audio(SILENCE_AUDIO_URL, autoplay=True))

# ============================================================
# INICIAR
# ============================================================

TOTAL_TIME = time.time() - START_TIME
print()
print("="*60)
print(f"✅ PRONTO! Tempo total: {TOTAL_TIME:.1f} segundos")
print("="*60)
print()
print("📋 COPIE A URL PÚBLICA ABAIXO")
print()

demo.queue().launch(share=True, debug=False)
